# Music2Vec Embedding Extractor

In [ ]:
!pip install torch librosa muq

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import hashlib
import torch
import librosa
from muq import MuQMuLan
from tqdm.notebook import tqdm

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

# Change these paths
AUDIO_DIR = '/content/drive/MyDrive/songs'
CACHE_DIR = '/content/drive/MyDrive/embeddings_cache'

os.makedirs(CACHE_DIR, exist_ok=True)

In [ ]:
def get_file_hash(filepath):
    hasher = hashlib.sha256()
    with open(filepath, 'rb') as f:
        buf = f.read()
        hasher.update(buf)
    return hasher.hexdigest()

print("Loading MuLan model...")
mulan = MuQMuLan.from_pretrained("OpenMuQ/MuQ-MuLan-large")
mulan = mulan.to(device).eval()
print("Model loaded.")

In [ ]:
audio_files = []
if os.path.exists(AUDIO_DIR):
    audio_files = [f for f in os.listdir(AUDIO_DIR) if f.endswith(('.wav', '.mp3', '.flac'))]

print(f"Found {len(audio_files)} audio files.")

for filename in tqdm(audio_files, desc="Extracting Embeddings"):
    audio_path = os.path.join(AUDIO_DIR, filename)
    file_hash = get_file_hash(audio_path)
    cache_path = os.path.join(CACHE_DIR, f"{file_hash}.pt")
    
    if os.path.exists(cache_path):
        continue
        
    try:
        wav, _ = librosa.load(audio_path, sr=24000)
        wavs = torch.tensor(wav).unsqueeze(0).to(device)
        
        with torch.no_grad():
            embeds = mulan(wavs=wavs)
            
        torch.save(embeds, cache_path)
    except Exception as e:
        print(f"Error processing {filename}: {e}")

print("Done!")